# Interfacer Init Data - Jupyter Notebook (Node.js Kernel)

> **Prerequisites**: This notebook requires the [ijavascript](https://github.com/n-riesco/ijavascript) kernel.
> Install with: `npm install -g ijavascript && ijsinstall`

This notebook injects test data into the Interfacer platform:
- **3 users** (Alice, Bob, Clara)
- **5 designs** with 3D model files
- **3 services**
- **5 products** (linked to designs, with 3 DPPs each)
- **Feedback** (reviews and comments)

## Cell 1: Configuration & Imports

In [ ]:
const { ApolloClient, InMemoryCache, HttpLink, gql } = require('@apollo/client/core');
const { zencode_exec } = require('zenroom');
const base64url = require('base64url');
const fs = require('fs');
const path = require('path');
const crypto = require('crypto');
require('cross-fetch/polyfill');

// Configuration
const BASE_URL = 'https://proxy.dpp-dev.ddns.dyne.org';
const ZENFLOWS_URL = `${BASE_URL}/zenflows/api`;
const ZENFLOWS_FILE_URL = `${BASE_URL}/zenflows/api/file`;
const DPP_URL = `${BASE_URL}/interfacer-dpp`;
const FEEDBACK_URL = 'http://localhost:8081';
const ZENFLOWS_ADMIN = 'b8b15ce1ef9b88ea87e66eab27faed749da3fbda525d0a38cc5a45a3e8319cc07603abe9c9d86c1e778adbf411982fee9bc5697e8bd72dcd2c75f9d34abd40d3';
const STL_FILE_PATH = '/Users/alcibiade/Desktop/incastro_mobile.stl';

// Results storage
const results = { users: [], designs: [], services: [], products: [], dpps: [], feedback: [] };

console.log('═══ Interfacer Init Data ═══');
console.log(`  Zenflows:  ${ZENFLOWS_URL}`);
console.log(`  DPP API:   ${DPP_URL}`);
console.log(`  Feedback:  ${FEEDBACK_URL}`);

## Cell 2: Helper Functions

In [ ]:
// Sign GraphQL request body with EdDSA
async function signGraphQL(body, eddsaPrivateKey, username) {
  const signScript = fs.readFileSync(
    path.join(__dirname, 'zenflows-crypto', 'src', 'sign_graphql.zen'),
    'utf8'
  );
  const gqlB64 = Buffer.from(body, 'utf8').toString('base64');
  const zenData = JSON.stringify({ gql: gqlB64 });
  const zenKeys = JSON.stringify({ keyring: { eddsa: eddsaPrivateKey } });
  const { result } = await zencode_exec(signScript, { data: zenData, keys: zenKeys });
  const parsed = JSON.parse(result);
  return {
    'zenflows-sign': parsed.eddsa_signature,
    'zenflows-user': username,
    'zenflows-hash': parsed.hash,
  };
}

// Zenflows request helper
async function zenflowsRequest(query, variables, auth = null) {
  const body = JSON.stringify({ query, variables });
  const headers = { 'Content-Type': 'application/json' };
  if (auth) {
    Object.assign(headers, await signGraphQL(body, auth.privateKey, auth.username));
  }
  if (ZENFLOWS_ADMIN) headers['zenflows-admin'] = ZENFLOWS_ADMIN;
  const res = await fetch(ZENFLOWS_URL, { method: 'POST', headers, body });
  const json = await res.json();
  if (json.errors) throw new Error(`GraphQL error: ${json.errors[0]?.message}`);
  return json.data;
}

// Sign REST body (DPP/Feedback)
async function signRESTBody(payload, privateKey, publicKey) {
  const signScript = `Scenario eddsa: sign a graph query\nGiven I have a 'base64' named 'gql'\nGiven I have the 'keyring'\n\nWhen I remove spaces in 'gql'\nand I compact ascii strings in 'gql'\n\nWhen I create the eddsa signature of 'gql'\nThen print 'eddsa signature' as 'base64'`;
  const gqlB64 = Buffer.from(payload || '', 'utf8').toString('base64');
  const zenData = JSON.stringify({ gql: gqlB64 });
  const zenKeys = JSON.stringify({ keyring: { eddsa: privateKey } });
  const { result } = await zencode_exec(signScript, { data: zenData, keys: zenKeys });
  return { 'did-sign': JSON.parse(result).eddsa_signature, 'did-pk': publicKey };
}

// File upload helpers
async function dppUploadFile(filePath, privateKey, publicKey) {
  const buf = fs.readFileSync(filePath);
  const hash = crypto.createHash('sha256').update(buf).digest('hex');
  const sigHeaders = await signRESTBody(hash, privateKey, publicKey);
  const FormData = require('form-data');
  const form = new FormData();
  form.append('file', buf, { filename: path.basename(filePath), contentType: 'application/sla' });
  const res = await fetch(`${DPP_URL}/upload`, { method: 'POST', headers: { ...sigHeaders }, body: form });
  if (!res.ok) throw new Error(`DPP upload failed: ${res.statusText}`);
  return res.json();
}

async function zenflowsUploadFile(filePath) {
  const buf = fs.readFileSync(filePath);
  const hash = base64url.fromBase64(crypto.createHash('sha512').update(buf).digest('base64'));
  const FormData = require('form-data');
  const form = new FormData();
  form.append(hash, buf, { filename: path.basename(filePath) });
  const res = await fetch(ZENFLOWS_FILE_URL, { method: 'POST', body: form });
  if (!res.ok) throw new Error(`File upload failed: ${res.statusText}`);
  return { name: path.basename(filePath), extension: path.extname(filePath).slice(1), hash, mimeType: 'image/png', size: buf.length };
}

console.log('✓ Helper functions defined');

## Cell 3: Generate Keys (Keypair Generation)

In [ ]:
// Load zenroom keypairoom script for client-side key generation
const keypairoomScript = fs.readFileSync(
  path.join(__dirname, 'zenflows-crypto', 'src', 'keypairoomClient-8-9-10-11-12.zen'),
  'utf8'
);

async function generateKeys(email, q1, q2, q3, q4, q5, hmac) {
  const zenData = JSON.stringify({
    userChallenges: { whereParentsMet: q1, nameFirstPet: q2, nameFirstTeacher: q3, whereHomeTown: q4, nameMotherMaid: q5 },
    username: email,
    'seedServerSideShard.HMAC': hmac,
  });
  const { result } = await zencode_exec(keypairoomScript, { data: zenData });
  const p = JSON.parse(result);
  return {
    eddsaPrivateKey: p.keyring.eddsa,
    eddsaPublicKey: p.eddsa_public_key,
    ethereumAddress: p.ethereum_address,
    reflowPublicKey: p.reflow_public_key,
    bitcoinPublicKey: p.bitcoin_public_key,
    ecdhPublicKey: p.ecdh_public_key,
  };
}
console.log('✓ Key generation function ready');

## Cell 4: Fetch Resource Specifications

In [ ]:
const QUERY_PROJECT_TYPES = `
  query GetProjectTypes {
    instanceVariables {
      specs {
        specProjectDesign { id name }
        specProjectProduct { id name }
        specProjectService { id name }
        specDpp { id name }
      }
      units { unitOne { id } }
    }
  }
`;

const specs = await zenflowsRequest(QUERY_PROJECT_TYPES);
const projectSpecs = {
  design: specs.invariantVariables?.specs?.specProjectDesign,
  product: specs.invariantVariables?.specs?.specProjectProduct,
  service: specs.invariantVariables?.specs?.specProjectService,
  dpp: specs.invariantVariables?.specs?.specDpp,
};
const unitOne = specs.invariantVariables?.units?.unitOne;

console.log('Specs loaded:');
console.log(`  Design:  ${projectSpecs.design?.id}`);
console.log(`  Product: ${projectSpecs.product?.id}`);
console.log(`  Service: ${projectSpecs.service?.id}`);
console.log(`  DPP:     ${projectSpecs.dpp?.id}`);
console.log(`  Unit:    ${unitOne?.id}`);

## Cell 5: Create 3 Users

In [ ]:
const SIGN_UP_MUTATION = `
  mutation SignUp($name:String!,$user:String!,$email:String!,$eddsaPublicKey:String!,$reflowPublicKey:String!,$ethereumAddress:String!,$ecdhPublicKey:String!,$bitcoinPublicKey:String!) {
    createPerson(person:{name:$name,user:$user,email:$email,eddsaPublicKey:$eddsaPublicKey,reflowPublicKey:$reflowPublicKey,ethereumAddress:$ethereumAddress,ecdhPublicKey:$ecdhPublicKey,bitcoinPublicKey:$bitcoinPublicKey}) {
      agent { id name user email }
    }
  }
`;

const REGISTER_MUTATION = `mutation RegisterUser($firstRegistration:Boolean!,$userData:JSONObject!) { keypairoomServer(firstRegistration:$firstRegistration,userData:$userData) }`;
const FETCH_SELF_QUERY = `query FetchSelf($email:String!,$pubkey:String!) { personCheck(email:$email,eddsaPublicKey:$pubkey) { name user email id } }`;

const users = [
  { name:'Alice Designer', username:'alice_designer', email:'alice.designer@example.com', c:{q1:'Paris',q2:'Rex',q3:'Smith',q4:'Berlin',q5:'Maria'} },
  { name:'Bob Maker', username:'bob_maker', email:'bob.maker@example.com', c:{q1:'London',q2:'Max',q3:'Johnson',q4:'Tokyo',q5:'Anna'} },
  { name:'Clara Reviewer', username:'clara_reviewer', email:'clara.reviewer@example.com', c:{q1:'Rome',q2:'Luna',q3:'Brown',q4:'Madrid',q5:'Sophia'} },
];

for (const u of users) {
  console.log(`Creating: ${u.name}`);
  const reg = await zenflowsRequest(REGISTER_MUTATION, { firstRegistration: true, userData: JSON.stringify({ email: u.email }) });
  const hmac = reg.keypairoomServer;
  const keys = await generateKeys(u.email, u.c.q1, u.c.q2, u.c.q3, u.c.q4, u.c.q5, hmac);
  const signup = await zenflowsRequest(SIGN_UP_MUTATION, {
    name: u.name, user: u.username, email: u.email,
    eddsaPublicKey: keys.eddsaPublicKey, reflowPublicKey: keys.reflowPublicKey,
    ethereumAddress: keys.ethereumAddress, ecdhPublicKey: keys.ecdhPublicKey, bitcoinPublicKey: keys.bitcoinPublicKey,
  });
  const agent = signup.createPerson.agent;
  await zenflowsRequest(FETCH_SELF_QUERY, { email: u.email, pubkey: keys.eddsaPublicKey });
  results.users.push({ id: agent.id, name: u.name, username: u.username, email: u.email, privateKey: keys.eddsaPrivateKey, publicKey: keys.eddsaPublicKey });
  console.log(`  ✓ Created: ${agent.id}`);
}
console.log(`\n${results.users.length} users created!`);

## Cell 6: Upload Test Assets (Image + 3D Model)

In [ ]:
// Create test image if needed
const testImagePath = path.join(__dirname, 'test_image.png');
if (!fs.existsSync(testImagePath)) {
  const png = Buffer.from('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAHggJ/PchI7wAAAABJRU5ErkJggg==', 'base64');
  fs.writeFileSync(testImagePath, png);
}
testImage = await zenflowsUploadFile(testImagePath);
console.log(`Test image uploaded: ${testImage.hash}`);

// Upload 3D model
let modelAttachment = null;
if (fs.existsSync(STL_FILE_PATH)) {
  const alice = results.users[0];
  modelAttachment = await dppUploadFile(STL_FILE_PATH, alice.privateKey, alice.publicKey);
  console.log(`3D model uploaded: ${modelAttachment.fileName} (${modelAttachment.id})`);
} else {
  console.log('⚠ STL file not found, designs will be created without 3D models');
}

## Cell 7: Create 5 Designs

In [ ]:
const CREATE_PROCESS = `mutation CreateProcess($name:String!) { createProcess(process:{name:$name}) { process { id } } }`;
const CREATE_PROJECT = `mutation CreateProject($name:String!,$note:String!,$metadata:JSONObject,$agent:ID!,$creationTime:DateTime!,$resourceSpec:ID!,$oneUnit:ID!,$images:[IFile!],$repo:String,$process:ID!,$license:String!,$tags:[URI!],$location:ID) {
  createEconomicEvent(event:{action:"produce",provider:$agent,receiver:$agent,outputOf:$process,hasPointInTime:$creationTime,resourceClassifiedAs:$tags,resourceConformsTo:$resourceSpec,resourceQuantity:{hasNumericalValue:1,hasUnit:$oneUnit},toLocation:$location,resourceMetadata:$metadata},newInventoriedResource:{name:$name,note:$note,images:$images,repo:$repo,license:$license}) {
    economicEvent { resourceInventoriedAs { id name } }
  }
}`;

// ---- Helper functions for filter tags (matching lib/tagging.ts) ----
function monotonicRangeTags(prefix,value,thresholds){
  if(!Number.isFinite(value))return[];
  const s=[...new Set(thresholds.filter(n=>Number.isFinite(n)))].sort((a,b)=>a-b);
  const f=v=>Number.isInteger(v)?String(v):String(v).replace(/\./g,'p');
  return [...new Set([...s.filter(t=>t<=value).map(t=>prefix+'-ge-'+f(t)),...s.filter(t=>t>=value).map(t=>prefix+'-le-'+f(t))])];
}
const slugify=s=>s.toLowerCase().trim().replace(/[^a-z0-9]+/g,'-').replace(/-+/g,'-').replace(/^-+/,'').replace(/-+$/,'');
const TH={rec:[0,10,20,30,40,50,60,70,80,90,100],pwr:[0,10,25,50,75,100,150,200,250,300,500,750,1000,1500,2000],en:[0,10,20,30,50,100,200,300,500,750,1000,1500,2000],co2:[0,0.5,1,1.5,2,2.5,3,3.5,5,7.5,10,15,20]};

// ---- Designs (with license tags) ----
const designs = [
  { name:'Modular Gear System', desc:'A parametric modular gear system for 3D printing.', link:'https://github.com/example/modular-gear', license:'CC-BY-SA-4.0', tags:['3d-printing','mechanical','parametric'] },
  { name:'Ergonomic Handle Grip', desc:'An ergonomic handle grip design.', link:'https://github.com/example/ergo-handle', license:'GPL-3.0', tags:['ergonomics','accessibility','3d-printing'] },
  { name:'Solar Panel Mount Bracket', desc:'Universal solar panel mounting bracket.', link:'https://github.com/example/solar-bracket', license:'CC0-1.0', tags:['solar','renewable-energy','mounting'] },
  { name:'Bicycle Cargo Rack', desc:'Lightweight bicycle cargo rack.', link:'https://github.com/example/bike-rack', license:'CC-BY-4.0', tags:['bicycle','transportation','cargo'] },
  { name:'Desktop Cable Organizer', desc:'Modular cable management system.', link:'https://github.com/example/cable-organizer', license:'MIT', tags:['organization','workspace','modular'] },
];

const alice = results.users[0];
for (const d of designs) {
  console.log(`Creating design: ${d.name}`);
  const p = await zenflowsRequest(CREATE_PROCESS, { name: `creation of ${d.name} by ${alice.username}` }, alice);
  const processId = p.createProcess.process.id;
  const models = modelAttachment ? [{
    contentType: modelAttachment.contentType, downloadUrl: `${DPP_URL}/file/${modelAttachment.id}`,
    extension: 'stl', fileName: modelAttachment.fileName, id: modelAttachment.id,
    mimeType: modelAttachment.contentType, name: modelAttachment.fileName,
    size: modelAttachment.size, storage: 'dpp', uploadedAt: modelAttachment.uploadedAt,
    url: `${DPP_URL}/file/${modelAttachment.id}`, checksum: modelAttachment.checksum,
  }] : [];
  const meta = JSON.stringify({ contributors:[], relations:[], remote:false, models });
  const r = await zenflowsRequest(CREATE_PROJECT, {
    name:d.name, note:d.desc, metadata:meta, agent:alice.id, creationTime:new Date().toISOString(),
    resourceSpec:projectSpecs.design.id, oneUnit:unitOne.id, images:[testImage], repo:d.link,
    process:processId, license:d.license, tags:[...d.tags.map(t=>'tag-'+slugify(t)),'license-'+slugify(d.license)],
  }, alice);
  const id = r.createEconomicEvent.resourceInventoriedAs.id;
  results.designs.push({ id, name:d.name, processId, models });
  console.log(`  ✓ Created: ${id}`);
}
console.log(`\n${results.designs.length} designs created!`);

## Cell 8: Create 3 Services

In [ ]:
const services = [
  { name:'3D Printing Consultation', desc:'Expert consultation for 3D printing.', link:'https://example.com/3d-consulting', license:'CC-BY-SA-4.0', tags:['consulting','3d-printing','education'], stype:['Fabrication','Learning & Education'], avail:['Booking Required','Weekends Available'] },
  { name:'Custom PCB Design Service', desc:'Professional PCB design and prototyping.', link:'https://example.com/pcb-design', license:'GPL-3.0', tags:['electronics','pcb','prototyping'], stype:['Fabrication','Space Access'], avail:['Available Now','Weekdays Only'] },
  { name:'Sustainable Packaging Consulting', desc:'Eco-friendly packaging solutions.', link:'https://example.com/eco-packaging', license:'CC0-1.0', tags:['sustainability','packaging','consulting'], stype:['Learning & Education'], avail:['Available Now','Weekends Available'] },
];

const bob = results.users[1];
for (const s of services) {
  console.log(`Creating service: ${s.name}`);
  const p = await zenflowsRequest(CREATE_PROCESS, { name: `creation of ${s.name} by ${bob.username}` }, bob);
  const processId = p.createProcess.process.id;
  const meta = JSON.stringify({ contributors:[], relations:[], remote:true, serviceFilters:{serviceType:s.stype||[],availability:s.avail||[]} });
  const svcTags = [...s.tags.map(t=>'tag-'+slugify(t)), ...(s.stype||[]).map(t=>'servicetype-'+slugify(t)), ...(s.avail||[]).map(a=>'availability-'+slugify(a))];
  const r = await zenflowsRequest(CREATE_PROJECT, {
    name:s.name, note:s.desc, metadata:meta, agent:bob.id, creationTime:new Date().toISOString(),
    resourceSpec:projectSpecs.service.id, oneUnit:unitOne.id, images:[testImage], repo:s.link,
    process:processId, license:s.license, tags:svcTags,
  }, bob);
  const id = r.createEconomicEvent.resourceInventoriedAs.id;
  results.services.push({ id, name:s.name, processId });
  console.log(`  ✓ Created: ${id}`);
}
console.log(`\n${results.services.length} services created!`);

## Cell 9: Create 5 Products (linked to designs)

In [ ]:
const CITE_PROJECT = `mutation cite($agent:ID!,$creationTime:DateTime!,$resource:ID!,$process:ID!,$unitOne:ID!) {
  createEconomicEvent(event:{action:"cite",inputOf:$process,provider:$agent,receiver:$agent,hasPointInTime:$creationTime,resourceInventoriedAs:$resource,resourceQuantity:{hasNumericalValue:1,hasUnit:$unitOne}}) {
    economicEvent { id }
  }
}`;

const products = [
  { name:'Premium Gear Set', desc:'High-precision modular gear set from recycled PLA.', link:'https://github.com/example/premium-gears', license:'CC-BY-SA-4.0', tags:['gears','robotics','mechanical'], di:0, pf:{categories:['Electronics','Tools'],powerCompatibility:['120V AC','Battery Powered'],replicability:['High'],recyclabilityPct:80,repairability:true,powerRequirementW:150,energyKwh:50,co2Kg:5} },
  { name:'ErgoGrip Pro Handle', desc:'Professional-grade ergonomic handle with dual-material construction.', link:'https://github.com/example/ergogrip-pro', license:'CC-BY-SA-4.0', tags:['ergonomics','professional','tools'], di:1, pf:{categories:['Tools','Wearables'],powerCompatibility:['Battery Powered','USB-C'],replicability:['Medium'],recyclabilityPct:65,repairability:true,powerRequirementW:10,energyKwh:20,co2Kg:1.5} },
  { name:'SunMount Universal Bracket', desc:'Heavy-duty solar panel bracket with adjustable angle.', link:'https://github.com/example/sunmount', license:'CC0-1.0', tags:['solar','renewable','outdoor'], di:2, pf:{categories:['Energy','Sustainability'],powerCompatibility:['220-240V AC','24V DC'],replicability:['Low','Medium'],recyclabilityPct:95,repairability:true,powerRequirementW:500,energyKwh:200,co2Kg:10} },
  { name:'Urban Cargo Rack XL', desc:'Extra-large bicycle cargo rack with pannier rails.', link:'https://github.com/example/urban-rack-xl', license:'CC-BY-4.0', tags:['bicycle','urban','transport'], di:3, pf:{categories:['Furniture','Sustainability'],powerCompatibility:['USB-C','12V DC'],replicability:['Medium'],recyclabilityPct:90,repairability:true,powerRequirementW:0,energyKwh:300,co2Kg:15} },
  { name:'DeskMate Cable System', desc:'Complete cable management kit for desks.', link:'https://github.com/example/deskmate', license:'MIT', tags:['desk','organization','office'], di:4, pf:{categories:['Education','Medical','Home renovation'],powerCompatibility:['12V DC','Battery Powered'],replicability:['High'],recyclabilityPct:70,repairability:true,powerRequirementW:75,energyKwh:100,co2Kg:2.5} },
];

for (const pr of products) {
  console.log(`Creating product: ${pr.name}`);
  const p = await zenflowsRequest(CREATE_PROCESS, { name: `creation of ${pr.name} by ${bob.username}` }, bob);
  const processId = p.createProcess.process.id;
  const designId = results.designs[pr.di].id;
  const allTags = [...pr.tags, ...pr.pf.categories.map(c => `category-${c}`), `recyclability-pct-${pr.pf.recyclabilityPct}`, 'repairability-true'];
  const meta = JSON.stringify({ contributors:[], relations:[], remote:false, design:designId, models:results.designs[pr.di].models, productFilters:pr.pf });
  const r = await zenflowsRequest(CREATE_PROJECT, {
    name:pr.name, note:pr.desc, metadata:meta, agent:bob.id, creationTime:new Date().toISOString(),
    resourceSpec:projectSpecs.product.id, oneUnit:unitOne.id, images:[testImage], repo:pr.link,
    process:processId, license:pr.license, tags:allTags,
  }, bob);
  const id = r.createEconomicEvent.resourceInventoriedAs.id;
  // Cite design
  await zenflowsRequest(CITE_PROJECT, { agent:bob.id, creationTime:new Date().toISOString(), resource:designId, process:processId, unitOne:unitOne.id }, bob);
  results.products.push({ id, name:pr.name, processId, designId });
  console.log(`  ✓ Created: ${id}`);
}
console.log(`${results.products.length} products created with linked designs!`);

## Cell 10: Create DPPs (3 per product = 15)

In [ ]:
const CREATE_DPP_RESOURCE = `mutation cdp($agent:ID!,$creationTime:DateTime!,$process:ID!,$resourceSpec:ID!,$unitOne:ID!,$dppUlid:String!,$name:String!,$note:String) {
  createEconomicEvent(event:{action:"produce",outputOf:$process,provider:$agent,receiver:$agent,hasPointInTime:$creationTime,resourceConformsTo:$resourceSpec,resourceQuantity:{hasNumericalValue:1,hasUnit:$unitOne},resourceMetadata:$dppUlid},newInventoriedResource:{name:$name,note:$note}) {
    economicEvent { id resourceInventoriedAs { id name metadata } }
  }
}`;

for (const product of results.products) {
  console.log(`Creating DPPs for: ${product.name}`);
  for (let i = 0; i < 3; i++) {
    const dppName = `DPP #${i+1} for ${product.name}`;
    try {
      // Create DPP document in DPP service
      const dp = await zenflowsRequest(CREATE_PROCESS, { name: `creation of ${dppName} by ${bob.username}` }, bob);
      const dpProcessId = dp.createProcess.process.id;
      const dppBody = {
        productId: product.id, batchType: i === 0 ? 'batch' : 'unit',
        batchId: `BATCH-${product.id.substring(0,8)}-${i+1}`,
        productOverview: { productName:{type:'Text',value:product.name}, productDescription:{type:'Text',value:`DPP #${i+1} for ${product.name}`} },
        environmentalImpact: { co2eEmissionsPerUnit:{type:'Number',value:12.5+i*3,units:'kg CO2e'}, energyConsumptionPerUnit:{type:'Number',value:45+i*10,units:'kWh'} },
        recyclability: { materialComposition:{type:'Text',value:'Recycled PLA 70%, Virgin PLA 30%'} },
        complianceAndStandards: { ceMarking:{type:'Text',value:'Yes'}, rohsCompliance:{type:'Text',value:'Yes'} },
        economicOperator: { companyName:{type:'Text',value:'Interfacer Demo Inc.'} },
      };
      const sigH = await signRESTBody(JSON.stringify(dppBody), bob.privateKey, bob.publicKey);
      const dppRes = await fetch(`${DPP_URL}/dpp`, { method:'POST', headers:{...sigH,'Content-Type':'application/json','x-user-id':bob.id}, body:JSON.stringify(dppBody) });
      if (!dppRes.ok) throw new Error(`DPP service error: ${dppRes.status}`);
      const { insertedID } = await dppRes.json();
      // Create DPP economic resource
      const dr = await zenflowsRequest(CREATE_DPP_RESOURCE, {
        agent:bob.id, creationTime:new Date().toISOString(), process:dpProcessId,
        resourceSpec:projectSpecs.dpp.id, unitOne:unitOne.id,
        dppUlid:JSON.stringify({dppServiceUlid:insertedID}), name:dppName, note:`DPP for ${product.name}`,
      }, bob);
      const dppResourceId = dr.createEconomicEvent.resourceInventoriedAs.id;
      // Cite DPP from product
      await zenflowsRequest(CITE_PROJECT, { agent:bob.id, creationTime:new Date().toISOString(), resource:dppResourceId, process:product.processId, unitOne:unitOne.id }, bob);
      results.dpps.push({ id:dppResourceId, dppUlid:insertedID, productId:product.id, productName:product.name, index:i });
      console.log(`  [${i+1}/3] ✓ ${insertedID} -> resource ${dppResourceId}`);
    } catch(e) { console.log(`  [${i+1}/3] ⚠ Failed: ${e.message}`); }
  }
}
console.log(`\n${results.dpps.length} DPPs created!`);

## Cell 11: Create Feedback (Reviews + Comments)

In [ ]:
const clara = results.users[2];

const reviewTargets = [results.products[0], results.products[2], results.products[4]];
for (const product of reviewTargets) {
  console.log(`Reviewing: ${product.name}`);
  try {
    // Review
    const reviewBody = JSON.stringify({ rating:4, content:`Great product! The ${product.name} is well-designed and works perfectly.` });
    const sigH = await signRESTBody(reviewBody, clara.privateKey, clara.publicKey);
    const rRes = await fetch(`${FEEDBACK_URL}/api/v1/projects/${product.id}/reviews`, { method:'POST', headers:{...sigH,'Content-Type':'application/json','x-user-id':clara.id}, body:reviewBody });
    if (rRes.ok) { const r = await rRes.json(); results.feedback.push({type:'review',productId:product.id,...r}); console.log(`  ✓ Review created`); }
    // Comment
    const commentBody = JSON.stringify({ content:`I've been using the ${product.name} and it's excellent. Build quality is great and docs are helpful.`, parent_id:null, attachments:null });
    const sigH2 = await signRESTBody(commentBody, clara.privateKey, clara.publicKey);
    const cRes = await fetch(`${FEEDBACK_URL}/api/v1/projects/${product.id}/comments`, { method:'POST', headers:{...sigH2,'Content-Type':'application/json','x-user-id':clara.id}, body:commentBody });
    if (cRes.ok) { const c = await cRes.json(); results.feedback.push({type:'comment',productId:product.id,...c}); console.log(`  ✓ Comment created`); }
  } catch(e) { console.log(`  ⚠ Failed: ${e.message}`); }
}
// Also review a design
if (results.designs[0]) {
  const d = results.designs[0];
  const rb = JSON.stringify({ rating:5, content:'Excellent parametric design! The modular gear system is incredibly versatile.' });
  const sh = await signRESTBody(rb, clara.privateKey, clara.publicKey);
  const rRes = await fetch(`${FEEDBACK_URL}/api/v1/projects/${d.id}/reviews`, { method:'POST', headers:{...sh,'Content-Type':'application/json','x-user-id':clara.id}, body:rb });
  if (rRes.ok) { const r = await rRes.json(); results.feedback.push({type:'review',productId:d.id,...r}); console.log(`  ✓ Design review created`); }
}
console.log(`\n${results.feedback.length} feedback items created!`);

## Cell 12: Summary & Export

In [ ]:
console.log('═══════════════════════════════════');
console.log('  DATA INJECTION COMPLETE');
console.log('═══════════════════════════════════');
console.log(`  Users:    ${results.users.length}`);
results.users.forEach(u => console.log(`    - ${u.name} (${u.id})`));
console.log(`  Designs:  ${results.designs.length}`);
results.designs.forEach(d => console.log(`    - ${d.name} (${d.id})`));
console.log(`  Services: ${results.services.length}`);
results.services.forEach(s => console.log(`    - ${s.name} (${s.id})`));
console.log(`  Products: ${results.products.length}`);
results.products.forEach(p => console.log(`    - ${p.name} (${p.id})`));
console.log(`  DPPs:     ${results.dpps.length}`);
console.log(`  Feedback: ${results.feedback.length}`);

fs.writeFileSync(path.join(__dirname, 'results.json'), JSON.stringify(results, null, 2));
console.log('\nResults saved to results.json');